# Speculative Decoding on TPU
### A NeurIPS 2026 Education Track submission: teaching materials notebook

**Concept:** Speculative decoding is an exact (lossless) LLM inference-acceleration technique.
A small, cheap **draft model** proposes several tokens ahead; the large **target model**
verifies them *in a single forward pass*. A rejection-sampling rule guarantees the
output distribution matches sampling from the target model alone, using far fewer
sequential target-model calls.

**Prerequisite knowledge:** advanced undergraduate or graduate familiarity with autoregressive language models,
softmax sampling, and basic probability (conditional distributions, rejection sampling).
Some exposure to `jax` (Sections 1-2) and `pytorch` (Section 3) is helpful but not
required. Every framework idiom used here is explained inline.

**Learning objectives.** By the end of this notebook you will be able to:
1. **Remember** the draft → verify → accept/reject loop that defines speculative decoding.
2. **Understand** *why* the rejection-sampling rule is mathematically lossless (same
   marginal distribution as standalone target-model sampling).
3. **Apply** the algorithm by implementing it from scratch in JAX.
4. **Analyze** the runtime trade-off between draft-model quality, the number of speculated
   tokens (`K`), and realized speedup on real hardware.
5. **Evaluate** how well the "free lunch" intuition holds by measuring empirical acceptance
   rates on real text and comparing against a token-level upper bound.
6. **Create** an extension of your own (see Exercises), e.g. a tree-structured draft
   (Medusa/EAGLE-style) instead of a single linear draft sequence.

**Companion papers (2023-2025, all in active use at the frontier):**
- Leviathan, Kalman & Matias, *Fast Inference from Transformers via Speculative Decoding*, ICML 2023.
- Chen et al., *Accelerating Large Language Model Decoding with Speculative Sampling*, arXiv:2302.01318, 2023 (DeepMind).
- Cai et al., *Medusa: Simple LLM Inference Acceleration Framework with Multiple Decoding Heads*, 2024.
- Li et al., *EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty*, ICML 2024.
- Li et al., *EAGLE-2: Faster Inference of Language Models with Dynamic Draft Trees*, EMNLP 2024.
- Li et al., *EAGLE-3: Scaling Up Inference Acceleration of Large Language Models via Training-Time Test*, arXiv:2503.01840, 2025.

---


## 0. Environment setup

**You do not need a TPU to do the core lesson.** Sections 1-2 implement and verify the
algorithm from scratch against tiny synthetic distributions, and run on a plain CPU in
seconds. Section 3 is an optional bonus that benchmarks real, openly available models to
make the wall-clock speedup concrete; it also runs on CPU (just slower), and speeds up
further if you have access to a GPU.

- **Without an accelerator:** run every cell in order. Sections 1-2 use JAX and detect
  CPU automatically (`jax.devices()`). Section 3 uses PyTorch and detects CPU/GPU via
  `torch.cuda.is_available()`; if it's running slowly on CPU, lower `K` or `N_STEPS` in
  the benchmark cell, or swap in smaller models.
- **With a TPU:** Sections 1-2 (JAX) run natively on TPU VMs provisioned through the
  [TPU Builders Program](https://sites.research.google/trc/) Getting Started Guide. SSH
  into your `ct6e-standard-4t` (or similar) flex-start instance and launch Jupyter, or
  run this notebook as a plain script. Section 3 (PyTorch) does not use the TPU natively
  without the separate `torch_xla` package; without it, Section 3 falls back to CPU or
  GPU on a TPU VM.

**Framework note (read before Section 3):** Section 3 uses PyTorch, not JAX/Flax.
Hugging Face `transformers` v5.0 (released January 2026) removed Flax support
entirely, and pinning an older `transformers` version to work around that runs into
a second problem: old `transformers`-Flax code depends on JAX internals (e.g.
`jax.core.get_opaque_trace_state`) that JAX itself has since removed. Chasing matching
old versions of two libraries at once is fragile and only gets worse over time.
PyTorch is the actively maintained, sole-supported backend in `transformers` now, so
Section 3 uses it directly. No version pinning is required.


In [ ]:
# If running fresh on a TPU VM, uncomment the line matching your TPU generation:
!pip install -U "jax[tpu]" -f https://storage.googleapis.com/jax-releases/libtpu_releases.html
# !pip install flax "transformers<5.0"  # transformers>=5.0 removed Flax support entirely

import jax
import jax.numpy as jnp
import numpy as np
import time

print("JAX version:", jax.__version__)
print("Devices visible to JAX:", jax.devices())
print("Backend:", jax.default_backend())  # 'tpu', 'gpu', or 'cpu'

## 1. The algorithm, in one picture

At each step, given a prefix `x`:

1. **Draft:** the small model `M_draft` autoregressively proposes `K` candidate tokens
   `x_{n+1}, ..., x_{n+K}`, and records its own probabilities `p_1, ..., p_K` for each.
2. **Verify:** the large model `M_target` runs **one** forward pass over the whole
   drafted prefix, producing target probabilities `q_1, ..., q_{K+1}` in parallel
   (the `K+1`-th distribution is the "bonus" token in case everything is accepted).
3. **Accept/Reject:** walk through the drafted tokens left to right. Accept token `i`
   with probability `min(1, q_i(x_i) / p_i(x_i))`. The **first** rejection stops the walk;
   at that position, sample a corrected token from the residual distribution
   `norm(max(0, q_i - p_i))` instead. If all `K` tokens are accepted, also emit the bonus
   token sampled from `q_{K+1}`.

The key theorem (Leviathan et al. 2023; Chen et al. 2023): **the marginal distribution of
each emitted token under this procedure is exactly `q`**, identical to what you'd get by
sampling from the target model alone, token by token. Nothing is approximated; you only
pay for it in wasted draft-model compute when a rejection happens early.


## 2. From-scratch implementation (toy distributions, pure JAX)

Before touching real neural nets, let's implement the accept/reject rule against
**synthetic categorical distributions** we fully control. This is the fastest way to
build intuition and to verify with your own eyes that the procedure is lossless. We'll
run thousands of trials and compare the empirical token distribution against `q` directly.


In [ ]:
key = jax.random.PRNGKey(0)
VOCAB = 8  # tiny vocabulary so we can print full distributions

def random_dist(key, vocab=VOCAB):
    logits = jax.random.normal(key, (vocab,))
    return jax.nn.softmax(logits)

k1, k2 = jax.random.split(key)
p_draft = random_dist(k1)   # the draft model's distribution over the next token
q_target = random_dist(k2)  # the target model's distribution over the next token

print("draft   p:", np.round(p_draft, 3))
print("target  q:", np.round(q_target, 3))

In [ ]:
def speculative_step(key, p, q):
    """One draft-verify-correct step for a SINGLE proposed token.
    Returns (token, accepted: bool)."""
    k_sample, k_accept, k_resid = jax.random.split(key, 3)

    # 1. draft proposes a token from p
    x = jax.random.categorical(k_sample, jnp.log(p))

    # 2. accept with probability min(1, q(x)/p(x))
    accept_prob = jnp.minimum(1.0, q[x] / p[x])
    accept = jax.random.uniform(k_accept) < accept_prob

    # 3. on rejection, sample from the residual distribution norm(max(0, q - p))
    residual = jnp.clip(q - p, 0.0)
    residual = residual / jnp.sum(residual)
    x_corrected = jax.random.categorical(k_resid, jnp.log(residual + 1e-12))

    token = jnp.where(accept, x, x_corrected)
    return token, accept

# Sanity check across many trials: does the EMITTED token distribution match q?
N = 200_000
keys = jax.random.split(jax.random.PRNGKey(1), N)
tokens, accepts = jax.vmap(lambda k: speculative_step(k, p_draft, q_target))(keys)

empirical = np.bincount(np.array(tokens), minlength=VOCAB) / N
print("target      q:", np.round(np.array(q_target), 4))
print("empirical draw:", np.round(empirical, 4))
print("max abs error :", np.max(np.abs(empirical - np.array(q_target))))
print("empirical accept rate:", float(jnp.mean(accepts)))

**What to notice:** the empirical distribution of emitted tokens matches `q_target`
to within Monte-Carlo noise, regardless of how different `p_draft` is from `q_target`. The
*speedup* comes from the accept rate. The more often `p_draft ≈ q_target` locally, the
more tokens you accept per target forward-pass. This is why draft-model *alignment* with
the target model (e.g. same tokenizer, distilled from the target, or fine-tuned on similar
data) matters more than draft-model raw accuracy.

Try changing `k1`/`k2` above, or manually constructing a `p_draft` that's close to
`q_target`, and re-run. Watch the accept rate rise.


## 3. Real models: measuring wall-clock speedup

Now we scale this up to real language models using PyTorch, the actively maintained backend in `transformers`, and benchmark standard autoregressive decoding vs. speculative decoding on real hardware.

We use two openly available causal LMs of very different size as the draft/target pair; swap in whatever pair you have quota for (e.g. a distilled or smaller sibling model as draft, its larger relative as target). Keeping the **same tokenizer** across draft and target is required, since acceptance is a token-level comparison.


In [ ]:
# Section 3 uses PyTorch directly. transformers dropped Flax support in v5.0, and
# pinning an old transformers version to work around that just trades one
# incompatibility for another against JAX's own internals (see the framework note
# above). PyTorch is the actively maintained backend, so there is no version to pin.

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DRAFT_NAME = "distilbert/distilgpt2"   # ~82M params
TARGET_NAME = "openai-community/gpt2-medium"  # ~355M params, same tokenizer family

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("No GPU detected. This section still runs, just more slowly. Native TPU "
          "execution needs the separate torch_xla package, which is not installed here.")

tokenizer = AutoTokenizer.from_pretrained(TARGET_NAME)
draft_model = AutoModelForCausalLM.from_pretrained(DRAFT_NAME).to(device).eval()
target_model = AutoModelForCausalLM.from_pretrained(TARGET_NAME).to(device).eval()

prompt = "The history of the Roman Empire begins with"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
print("Prompt tokens:", input_ids.shape)


In [ ]:
@torch.no_grad()
def draft_propose(model, ids, k_tokens):
    """Greedily-sampled draft continuation of length k_tokens. Returns the drafted
    ids AND the full draft distribution at each step (needed later to build the
    residual distribution, not just the probability of the sampled token)."""
    dists = []
    for _ in range(k_tokens):
        logits = model(ids).logits[0, -1]
        p = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(p, num_samples=1)
        dists.append(p)
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return ids, torch.stack(dists)


@torch.no_grad()
def target_verify(model, ids, prefix_len, k_tokens):
    """One parallel forward pass over the drafted sequence; returns target probs
    for each of the k_tokens drafted positions (+1 bonus)."""
    logits = model(ids).logits[0]  # (seq_len, vocab)
    # position i in logits predicts token i+1
    out_probs = torch.softmax(logits[prefix_len - 1: prefix_len - 1 + k_tokens + 1], dim=-1)
    return out_probs


@torch.no_grad()
def speculative_generate(prompt_ids, k_tokens, n_steps):
    """Implements the full draft-verify-accept/reject-correct loop, matching
    Leviathan et al. (2023) / Chen et al. (2023). On the first rejection we sample
    a REPLACEMENT token from the residual distribution norm(max(0, q - p)); we do
    not keep the rejected draft token. On full acceptance we sample the bonus
    token from the target's own distribution. Both cases require an explicit
    extra sample; neither can be skipped without breaking the exactness guarantee."""
    ids = prompt_ids.clone()
    accepted_total, drafted_total = 0, 0
    for _ in range(n_steps):
        prefix_len = ids.shape[1]
        draft_ids, draft_dists = draft_propose(draft_model, ids, k_tokens)
        target_probs = target_verify(target_model, draft_ids, prefix_len, k_tokens)

        n_accepted = 0
        next_token = None
        for i in range(k_tokens):
            tok = draft_ids[0, prefix_len + i]
            p_vec, q_vec = draft_dists[i], target_probs[i]
            accept_prob = torch.clamp(q_vec[tok] / (p_vec[tok] + 1e-12), max=1.0)
            if torch.rand(()).item() < accept_prob.item():
                n_accepted += 1
                continue
            # rejected: sample the replacement token from the residual distribution
            residual = torch.clamp(q_vec - p_vec, min=0.0)
            residual = residual / residual.sum()
            next_token = torch.multinomial(residual, num_samples=1)
            break
        else:
            # every drafted token was accepted: sample the bonus token from q directly
            next_token = torch.multinomial(target_probs[k_tokens], num_samples=1)

        accepted_total += n_accepted
        drafted_total += k_tokens
        kept = draft_ids[:, :prefix_len + n_accepted]
        ids = torch.cat([kept, next_token.view(1, 1)], dim=1)
    return ids, accepted_total, drafted_total


@torch.no_grad()
def baseline_generate(prompt_ids, n_tokens):
    ids = prompt_ids.clone()
    for _ in range(n_tokens):
        logits = target_model(ids).logits[0, -1]
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return ids


In [ ]:
K = 4          # number of tokens the draft model speculates per round
N_STEPS = 15    # speculative rounds
torch.manual_seed(42)

# Baseline: token-by-token target-only decoding
t0 = time.time()
baseline_ids = baseline_generate(input_ids, K * N_STEPS)
t_baseline = time.time() - t0

# Speculative decoding
t0 = time.time()
spec_ids, accepted, drafted = speculative_generate(input_ids, K, N_STEPS)
t_spec = time.time() - t0

print(f"Device: {device}")
print(f"Baseline wall time   : {t_baseline:.2f}s for {K*N_STEPS} tokens")
print(f"Speculative wall time: {t_spec:.2f}s, empirical accept rate = {accepted/drafted:.2%}")
print(f"Speedup: {t_baseline / max(t_spec, 1e-9):.2f}x")
print()
print("Baseline output   :", tokenizer.decode(baseline_ids[0]))
print("Speculative output:", tokenizer.decode(spec_ids[0]))


**Notes for benchmarking:**
- The first call to each model on GPU pays for CUDA kernel warmup and lazy initialization. Always run a warm-up call before timing, or report only post-warm-up timings.
- Per-step overhead (the Python loop, host-device sync) dominates for tiny models; speculative decoding's advantage grows as the **target model gets larger relative to the draft model** and as **sequence lengths grow**. Try swapping in a bigger target (e.g. `gpt2-large` or `gpt2-xl`) if your hardware allows, and re-run.
- This reference implementation favors clarity over throughput (Python-level loop over `K`, no `torch.compile`). A production implementation (e.g. vLLM) fuses the verify step and batches the accept/reject logic. See Exercise 3.
- Native TPU execution needs the separate `torch_xla` package, which is not installed by default here. Without it, this section runs on GPU if available, otherwise CPU.


## 4. Exercises

1. **Check your understanding (no code, no hardware needed).** Using the toy
   distributions `p_draft` and `q_target` printed in Section 2, compute by hand the
   acceptance probability `min(1, q_target[x] / p_draft[x])` for whichever token index
   `x` has the highest value under `p_draft`. Then, in a sentence or two, explain why the
   residual-distribution correction step is required for the output to be lossless,
   rather than just a nice-to-have detail.
2. **Acceptance rate vs. draft/target similarity.** Replace `DRAFT_NAME` with a model
   *fine-tuned from the same checkpoint family as the target* (a common real-world setup)
   and compare the empirical acceptance rate to the mismatched pair above. Plot acceptance
   rate vs. `K`.
3. **Speed up the reference loop.** Wrap `draft_propose` and `target_verify` in
   `torch.compile`, and re-measure the speedup. What changes, and why does graph
   compilation reduce the overhead of the Python-level loop here?
4. **Tree-structured drafting (open-ended, Medusa/EAGLE-style).** Instead of one linear
   draft sequence, propose a *tree* of candidate continuations (e.g. top-2 branches at
   each of the K positions) and verify the whole tree in a single target forward pass
   using a tree-attention mask. Compare achievable speedup against the linear baseline.
5. **Where does speculative decoding break down?** Construct a prompt/task where the
   draft model's distribution is adversarially different from the target's (e.g. code
   the draft model has never seen). Explain, from the algorithm's guarantee, why
   correctness is preserved even though speed is not.


## 5. Further reading

- Leviathan, Kalman & Matias (2023). *Fast Inference from Transformers via Speculative Decoding.* ICML.
- Chen, Borgeaud, Irving, Lespiau, Sifre & Jumper (2023). *Accelerating Large Language Model Decoding with Speculative Sampling.* arXiv:2302.01318.
- Cai, Li, Geng, Peng, Lee, Chen & Dao (2024). *Medusa: Simple LLM Inference Acceleration Framework with Multiple Decoding Heads.* ICML.
- Li, Wei, Zhang & Zhang (2024). *EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty.* ICML.
- Li, Wei, Zhang & Zhang (2024). *EAGLE-2: Faster Inference of Language Models with Dynamic Draft Trees.* EMNLP.
- Li, Wei, Zhang & Zhang (2025). *EAGLE-3: Scaling Up Inference Acceleration of Large Language Models via Training-Time Test.* arXiv:2503.01840.
- Xia et al. (2024). *Unlocking Efficiency in Large Language Model Inference: A Comprehensive Survey of Speculative Decoding.* Findings of ACL.

**TPU/JAX resources (TPU Builders Program):**
- *How to Think About TPUs*: https://jax-ml.github.io/scaling-book/
- Core JAX documentation: https://docs.jax.dev/
